In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

matplotlib.rcParams.update({
    'font.family':    'serif',
    'font.size':       8,
    'axes.titlesize':  8,
    'axes.labelsize':  7.5,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'savefig.dpi':     300,
    'savefig.bbox':   'tight',
})

ITER_COLORS = ['#2166ac', '#4dac26', '#d6604d', '#762a83']
NAT_COLOR   = '#888888'
OUTPUT_DIR  = Path('outputs')

# ── Load results ───────────────────────────────────────────────────────────
df           = pd.read_csv(OUTPUT_DIR / 'smgil_mfp_temporal_validation.csv')
df_amenable  = df.copy()   # 0% non-amenable
nat_med_nut  = df_amenable['d_nut_natural'].median()
nat_med_food = df_amenable['d_food_natural'].median()

# ── Wilcoxon tests ─────────────────────────────────────────────────────────
wilcoxon_results = {}
for metric in ['d_nut', 'd_food']:
    nat_col = metric + '_natural'
    for r in [1, 2, 3, 4]:
        rec_col = metric + '_r' + str(r)
        if rec_col not in df_amenable.columns:
            continue
        merged = df_amenable[[nat_col, rec_col]].dropna()
        if len(merged) < 10:
            continue
        W_s, p_s = stats.wilcoxon(merged[rec_col], merged[nat_col], alternative='less')
        wilcoxon_results[(metric, r)] = {
            'n': len(merged), 'W': W_s, 'p': p_s,
            'med_rec': merged[rec_col].median(),
            'med_nat': merged[nat_col].median(),
        }

r1_nut  = df_amenable[['d_nut_natural',  'd_nut_r1']].dropna()
r1_food = df_amenable[['d_food_natural', 'd_food_r1']].dropna()
pct_below_diag_nut  = (r1_nut['d_nut_r1']   < r1_nut['d_nut_natural']).mean()   * 100
pct_below_diag_food = (r1_food['d_food_r1'] < r1_food['d_food_natural']).mean() * 100

print('Data loaded OK.')
print(f'  N = {len(df_amenable)}')
print(f'  Natural nutrient median: {nat_med_nut:.3f}')
print(f'  r=1 nutrient median:     {wilcoxon_results[("d_nut", 1)]["med_rec"]:.3f}')
print(f'  W={wilcoxon_results[("d_nut", 1)]["W"]:.0f}  p={wilcoxon_results[("d_nut", 1)]["p"]:.3e}')

In [ ]:
# ── Build figure ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(7.16, 5.8))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.52, wspace=0.40)

ax_scatter = fig.add_subplot(gs[0, 0])
ax_box_nut = fig.add_subplot(gs[0, 1])
ax_box_fod = fig.add_subplot(gs[0, 2])
ax_ratio_n = fig.add_subplot(gs[1, 0])
ax_ratio_f = fig.add_subplot(gs[1, 1])
ax_frac    = fig.add_subplot(gs[1, 2])

# Panel (a): scatter r=1 vs natural
nat_v = r1_nut['d_nut_natural'].values
rec_v = r1_nut['d_nut_r1'].values
lim   = max(nat_v.max(), rec_v.max()) * 1.06
ax_scatter.scatter(nat_v, rec_v, s=10, alpha=0.50,
                   color=ITER_COLORS[0], edgecolors='none')
ax_scatter.plot([0, lim], [0, lim], '--', color='#555', lw=0.8)
ax_scatter.set_xlabel('Natural (norm. L2)')
ax_scatter.set_ylabel('Rec. $r{=}1$ (norm. L2)')
ax_scatter.set_title('(a) Scatter: $r{=}1$ vs natural')
ax_scatter.text(0.05, 0.93,
                str(int(round(pct_below_diag_nut))) + '% below diagonal',
                transform=ax_scatter.transAxes, fontsize=6.5, color='#222')
ax_scatter.set_xlim(0, lim); ax_scatter.set_ylim(0, lim)
ax_scatter.spines[['top', 'right']].set_visible(False)

# Panel (b): nutrient distance box plots
r_list     = [r for r in [1, 2, 3, 4] if 'd_nut_r' + str(r) in df_amenable.columns]
nut_cols   = ['d_nut_natural'] + ['d_nut_r' + str(r) for r in r_list]
nut_labels = ['Natural'] + ['$r{=}' + str(r) + '$' for r in r_list]
nut_data   = [df_amenable[c].dropna().values for c in nut_cols]
nut_colors = [NAT_COLOR] + ITER_COLORS[:len(r_list)]

bp = ax_box_nut.boxplot(nut_data, patch_artist=True,
                        medianprops=dict(color='black', lw=1.2),
                        flierprops=dict(marker='.', ms=2, alpha=0.35),
                        whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
for patch, color in zip(bp['boxes'], nut_colors):
    patch.set_facecolor(color); patch.set_alpha(0.70)
ax_box_nut.set_xticklabels(nut_labels, fontsize=6.5)
ax_box_nut.axhline(nat_med_nut, color=NAT_COLOR, ls=':', lw=0.9)
ax_box_nut.set_ylabel('Nutrient dist. (norm. L2)')
ax_box_nut.set_title('(b) Nutrient distance by iteration')
ax_box_nut.spines[['top', 'right']].set_visible(False)

# Panel (c): food distance box plots
fod_cols   = ['d_food_natural'] + ['d_food_r' + str(r) for r in r_list]
fod_labels = ['Natural'] + ['$r{=}' + str(r) + '$' for r in r_list]
fod_data   = [df_amenable[c].dropna().values for c in fod_cols]
fod_colors = [NAT_COLOR] + ITER_COLORS[:len(r_list)]

bp2 = ax_box_fod.boxplot(fod_data, patch_artist=True,
                         medianprops=dict(color='black', lw=1.2),
                         flierprops=dict(marker='.', ms=2, alpha=0.35),
                         whiskerprops=dict(lw=0.7), capprops=dict(lw=0.7))
for patch, color in zip(bp2['boxes'], fod_colors):
    patch.set_facecolor(color); patch.set_alpha(0.70)
ax_box_fod.set_xticklabels(fod_labels, fontsize=6.5)
ax_box_fod.axhline(nat_med_food, color=NAT_COLOR, ls=':', lw=0.9)
ax_box_fod.set_ylabel('Food dist. (servings)')
ax_box_fod.set_title('(c) Food distance by iteration')
ax_box_fod.spines[['top', 'right']].set_visible(False)

# Panel (d): nutrient ratio distribution r=1
ratio_n = (r1_nut['d_nut_r1'] /
           r1_nut['d_nut_natural'].replace(0, np.nan)).dropna()
ratio_n = ratio_n[np.isfinite(ratio_n) & (ratio_n < 4)]
ax_ratio_n.hist(ratio_n, bins=28, color=ITER_COLORS[0],
                edgecolor='white', lw=0.3, alpha=0.85)
ax_ratio_n.axvline(1.0, color='black', ls='--', lw=0.9)
ax_ratio_n.axvline(ratio_n.median(), color=ITER_COLORS[0], ls='-', lw=1.1, alpha=0.9)
ax_ratio_n.set_xlabel('$d_{\\mathrm{rec}} / d_{\\mathrm{nat}}$ (nutrient)')
ax_ratio_n.set_ylabel('Users')
ax_ratio_n.set_title(
    '(d) Nutrient ratio ($r{=}1$)\n'
    + str(int(round((ratio_n < 1).mean() * 100))) + '% $<$ 1.0'
    + '  |  median=' + str(round(ratio_n.median(), 2))
)
ax_ratio_n.spines[['top', 'right']].set_visible(False)

# Panel (e): food ratio distribution r=1
ratio_f = (r1_food['d_food_r1'] /
           r1_food['d_food_natural'].replace(0, np.nan)).dropna()
ratio_f = ratio_f[np.isfinite(ratio_f) & (ratio_f < 4)]
ax_ratio_f.hist(ratio_f, bins=28, color='#2ca02c',
                edgecolor='white', lw=0.3, alpha=0.85)
ax_ratio_f.axvline(1.0, color='black', ls='--', lw=0.9)
ax_ratio_f.axvline(ratio_f.median(), color='#2ca02c', ls='-', lw=1.1, alpha=0.9)
ax_ratio_f.set_xlabel('$d_{\\mathrm{rec}} / d_{\\mathrm{nat}}$ (food)')
ax_ratio_f.set_ylabel('Users')
ax_ratio_f.set_title(
    '(e) Food ratio ($r{=}1$)\n'
    + str(int(round((ratio_f < 1).mean() * 100))) + '% $<$ 1.0'
    + '  |  median=' + str(round(ratio_f.median(), 2))
)
ax_ratio_f.spines[['top', 'right']].set_visible(False)

# Panel (f): % below natural median
frac_nut  = [(df_amenable['d_nut_r'  + str(r)].dropna() < nat_med_nut ).mean() * 100
             for r in r_list]
frac_food = [(df_amenable['d_food_r' + str(r)].dropna() < nat_med_food).mean() * 100
             for r in r_list if 'd_food_r' + str(r) in df_amenable.columns]

ax_frac.plot(r_list, frac_nut,  'o-', color=ITER_COLORS[0], lw=1.3, ms=5, label='Nutrient')
ax_frac.plot(r_list[:len(frac_food)], frac_food, 's-', color='#2ca02c',
             lw=1.3, ms=5, label='Food')
ax_frac.set_xticks(r_list)
ax_frac.set_xlabel('Iteration $r$')
ax_frac.set_ylabel('% below natural median')
ax_frac.set_title('(f) % below natural median')
ax_frac.set_ylim(0, 105)
ax_frac.legend(frameon=False, loc='lower left')
ax_frac.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    'SGIO MyFitnessPal Temporal-Split Validation ($N=' + str(len(df_amenable)) + '$)',
    fontsize=9, fontweight='bold', y=1.01,
)

plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'smgil_mfp_validation_final.pdf'))
fig.savefig(str(OUTPUT_DIR / 'smgil_mfp_validation_final.png'))
plt.show()
print('Saved smgil_mfp_validation_final.{pdf,png}')

In [ ]:
# ── LaTeX table ────────────────────────────────────────────────────────────
def sig_stars(p):
    if   p < 0.001: return '$^{***}$'
    elif p < 0.01:  return '$^{**}$'
    elif p < 0.05:  return '$^{*}$'
    else:           return '$^{\\phantom{*}}$'

latex = (
    '\\begin{table}[t]\n'
    '\\caption{SGIO MyFitnessPal Temporal-Split Validation Results ($N=200$)}\n'
    '\\label{tab:mfp_results}\n'
    '\\begin{center}\n'
    '\\begin{tabular}{lcccc}\n'
    '\\toprule\n'
    ' & \\multicolumn{2}{c}{\\textbf{Nutrient Distance}}\n'
    ' & \\multicolumn{2}{c}{\\textbf{Food Distance (srv)}} \\\\\n'
    '\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}\n'
    ' & Median & \\%\\,below$^{\\dagger}$ & Median & \\%\\,below$^{\\dagger}$ \\\\\n'
    '\\midrule\n'
    'Natural (train$\\to$test) & '
    + str(round(nat_med_nut, 3)) + ' & --- & '
    + str(round(nat_med_food, 3)) + ' & --- \\\\\n'
    '\\midrule\n'
)

for r in [1, 2, 3, 4]:
    rn  = wilcoxon_results.get(('d_nut',  r), {})
    rf  = wilcoxon_results.get(('d_food', r), {})
    n_r = rn.get('n', '---')

    med_nut  = str(round(rn['med_rec'], 3)) + sig_stars(rn['p']) if rn else '---'
    med_food = str(round(rf['med_rec'], 3)) + sig_stars(rf['p']) if rf else '---'

    pct_nut_val  = (df_amenable['d_nut_r'  + str(r)].dropna() < nat_med_nut ).mean() * 100
    pct_food_val = (df_amenable['d_food_r' + str(r)].dropna() < nat_med_food).mean() * 100 \
                   if 'd_food_r' + str(r) in df_amenable.columns else None

    pct_nut  = str(int(round(pct_nut_val)))  + '\\%'
    pct_food = str(int(round(pct_food_val))) + '\\%' if pct_food_val is not None else '---'

    latex += (
        '$r{=}' + str(r) + '$ ($n{=}' + str(n_r) + '$) & '
        + med_nut  + ' & ' + pct_nut  + ' & '
        + med_food + ' & ' + pct_food + ' \\\\\n'
    )

latex += (
    '\\bottomrule\n'
    '\\multicolumn{5}{l}{%\n'
    '  \\footnotesize $^{\\dagger}$\\% users below cohort natural median.}\\\\\n'
    '\\multicolumn{5}{l}{%\n'
    '  \\footnotesize $^{***}p{<}0.001$; one-sided Wilcoxon signed-rank test.}\n'
    '\\end{tabular}\n'
    '\\end{center}\n'
    '\\end{table}'
)

print(latex)
with open(OUTPUT_DIR / 'mfp_final_table.tex', 'w') as f:
    f.write(latex)
print('\nSaved outputs/mfp_final_table.tex')

In [ ]:
# ── Key numbers + paper paragraph ─────────────────────────────────────────
rn1 = wilcoxon_results[('d_nut', 1)]
rf1 = wilcoxon_results[('d_food', 1)]

print('=' * 65)
print('KEY NUMBERS FOR PAPER')
print('=' * 65)
print(f'  N cohort:                {len(df_amenable)}')
print(f'  Non-amenable:            0 (0%)')
print(f'  Natural nutrient median: {nat_med_nut:.3f}')
print(f'  r=1 nutrient median:     {rn1["med_rec"]:.3f}')
print(f'  Reduction:               {100*(1 - rn1["med_rec"]/nat_med_nut):.0f}%')
print(f'  W={rn1["W"]:.0f}  p={rn1["p"]:.3e}')
print(f'  % below diagonal:        {pct_below_diag_nut:.0f}%')
print(f'  % below nat. median:     {(df_amenable["d_nut_r1"] < nat_med_nut).mean()*100:.0f}%')
print()
print(f'  Natural food median:     {nat_med_food:.3f}')
print(f'  r=1 food median:         {rf1["med_rec"]:.3f}')
print(f'  W={rf1["W"]:.0f}  p={rf1["p"]:.3e}')
print(f'  % below diagonal:        {pct_below_diag_food:.0f}%')

print()
print('=' * 65)
print('PAPER PARAGRAPH — paste into Section 4.3 (MFP Results)')
print('=' * 65)
print(r"""
On the MFP cohort ($n{=}200$), all users received at least one
recommendation (0\% non-amenable rate), in contrast to the 44\%
non-amenable rate observed on NHANES. This reflects the broader
dietary repertoire captured across multi-day food logs: the larger
augmented item space ($|\mathcal{F}_{\mathrm{aug}}|\approx 1{,}700$
vs.\ 135 in NHANES) provides the optimizer with sufficient
substitution flexibility to satisfy DASH constraints at near-zero
marginal cost for all users. SGIO recommendations at $r{=}1$
achieve a median normalized nutrient distance of 1.637 from the
training-period mean, compared to a natural train-to-test median
of 3.092 --- a 47\% reduction (Wilcoxon $W{=}153$, $p{<}0.001$).
In food-item space, 95\% of $r{=}1$ recommendations fall below
the individual's natural test-period distance, and 92\% fall
below the cohort natural median. The marginal cost of satisfying
additional DASH constraints is negligible across iterations
($\Delta D^{\mathbf{W}}_{r=1\to r=4} \approx 0$), indicating
that MFP users can achieve full DASH compliance at essentially
no additional behavioral burden beyond the first-iteration
recommendation; accordingly, $r{=}1$ is the recommended
deployment point for this data modality.
""")